In [1]:
using Pkg
Pkg.activate("C:/Users/selha/Desktop/MAAAI/environment")  # <-- CAMBIAR A TU RUTA DE ENV

  Activating project at `C:\Users\selha\Desktop\MAAAI\environment`


In [2]:
Pkg.instantiate()   # solo necesario la primera vez 

In [3]:
using CSV, DataFrames, Statistics, Random
using MLJ
using MLJModelInterface
import MLJBase: transform
const MMI = MLJModelInterface
using HypothesisTests
using DataFramesMeta
using StatsBase
using GLM, StatsModels
using Plots
using Glob
using Flux
using MLJScikitLearnInterface
using MultivariateStats

# PREPARACIÓN DE DATOS
## 1. Carga y unificación de datos

Los datos originales están distribuidos en múltiples ficheros CSV, uno por cada sujeto.  
El objetivo de este bloque es:

- Buscar todos los CSV dentro del directorio raíz.
- Leerlos en memoria de forma homogénea.
- Concatenarlos en un único DataFrame consolidado.
- Guardar `dataset_consolidado.csv` para usarlo en el resto de la práctica.


In [4]:
DATA_ROOT = "C:\\Users\\selha\\Desktop\\MAAAI\\datasets"

function find_all_csv_files(root_dir)
    csv_files = String[]
    for (root, dirs, files) in walkdir(root_dir)
        for file in files
            if endswith(file, ".csv")
                push!(csv_files, joinpath(root, file))
            end
        end
    end
    return csv_files
end

csv_files = find_all_csv_files(DATA_ROOT)
println("Archivos encontrados: ", length(csv_files))

# Leer todos los CSV
dfs = DataFrame[]
for file in csv_files
    try
        df_tmp = CSV.read(file, DataFrame; normalizenames=true)
        push!(dfs, df_tmp)
        println("✓ Leído: ", basename(file), " (", nrow(df_tmp), " filas)")
    catch e
        @warn "No se pudo leer: $file" exception=(e, catch_backtrace())
    end
end

# Consolidar
df_all = vcat(dfs...)
println("\nDataset consolidado: ", nrow(df_all), " filas, ", ncol(df_all), " columnas")

# Guardar
mkpath("data_processed")
CSV.write("data_processed/dataset_consolidado.csv", df_all)
println(" Guardado: data_processed/dataset_consolidado.csv")

Archivos encontrados: 30
✓ Leído: Sujeto_01.csv (347 filas)
✓ Leído: Sujeto_05.csv (302 filas)
✓ Leído: Sujeto_07.csv (308 filas)
✓ Leído: Sujeto_11.csv (316 filas)
✓ Leído: Sujeto_03.csv (341 filas)
✓ Leído: Sujeto_09.csv (288 filas)
✓ Leído: Sujeto_23.csv (372 filas)
✓ Leído: Sujeto_25.csv (409 filas)
✓ Leído: Sujeto_15.csv (328 filas)
✓ Leído: Sujeto_17.csv (368 filas)
✓ Leído: Sujeto_21.csv (408 filas)
✓ Leído: Sujeto_13.csv (327 filas)
✓ Leído: Sujeto_19.csv (360 filas)
✓ Leído: Sujeto_27.csv (376 filas)
✓ Leído: Sujeto_29.csv (344 filas)
✓ Leído: Sujeto_02.csv (302 filas)
✓ Leído: Sujeto_04.csv (317 filas)
✓ Leído: Sujeto_06.csv (325 filas)
✓ Leído: Sujeto_08.csv (281 filas)
✓ Leído: Sujeto_10.csv (294 filas)
✓ Leído: Sujeto_12.csv (320 filas)
✓ Leído: Sujeto_14.csv (323 filas)
✓ Leído: Sujeto_16.csv (366 filas)
✓ Leído: Sujeto_18.csv (364 filas)
✓ Leído: Sujeto_20.csv (354 filas)
✓ Leído: Sujeto_22.csv (321 filas)
✓ Leído: Sujeto_24.csv (381 filas)
✓ Leído: Sujeto_26.csv (392 fi

## 2. Resumen del conjunto de datos

En esta sección obtenemos una descripción básica del dataset consolidado:

- Número total de instancias (filas)
- Número total de variables
- Número de individuos (`subject`)
- Número de clases de salida (`Activity`)


In [5]:
# Cargar dataset consolidado desde data_processed
df = CSV.read("data_processed/dataset_consolidado.csv", DataFrame)

num_variables_totales = ncol(df)
num_instancias = nrow(df)

# Número de individuos
if "subject" in names(df)
    num_individuos = length(unique(df.subject))
else
    @warn "No se encontró la columna 'subject'; no se puede calcular el número de individuos."
    num_individuos = missing
end

# Número de clases de salida
if "Activity" in names(df)
    num_clases_salida = length(unique(df.Activity))
else
    @warn "No se encontró la columna 'Activity'; no se puede calcular el número de clases de salida."
    num_clases_salida = missing
end

# Variables de entrada (todas excepto subject + Activity)
num_features = num_variables_totales - 2

println("\n=== Resumen del dataset ===")
println("Número total de variables (incluyendo subject y Activity): ", num_variables_totales)
println("Número de variables de características: ", num_features)
println("Número de instancias: ", num_instancias)
println("Número de individuos: ", num_individuos)
println("Número de clases de salida: ", num_clases_salida)
println("=============================================")


=== Resumen del dataset ===
Número total de variables (incluyendo subject y Activity): 563
Número de variables de características: 561
Número de instancias: 10299
Número de individuos: 30
Número de clases de salida: 6


## 3. Análisis de valores ausentes

En esta sección calculamos:

- El **porcentaje de valores nulos por variable**  
- El **porcentaje total de valores nulos** en el dataset  

Esto permite entender la magnitud del problema de valores faltantes y justificar
posteriormente el método de imputación empleado (subject-wise con media/mediana).

In [6]:
# Cargar dataset consolidado desde data_processed
df = CSV.read("data_processed/dataset_consolidado.csv", DataFrame)

# Análisis de valores ausentes
n_rows = nrow(df)
n_cols = ncol(df)

# Porcentaje de nulos por columna
porc_nulos_col = Dict{String, Float64}()

for col in names(df)
    n_missing = count(ismissing, df[!, col])
    porc_nulos_col[col] = 100 * n_missing / n_rows
end

# Porcentaje total de nulos en todo el dataset
total_missing = sum(count(ismissing, df[!, col]) for col in names(df))
total_values = n_rows * n_cols
porc_total_missing = 100 * total_missing / total_values

println("=== Porcentaje de valores nulos por columna ===")
for (col, pct) in sort(collect(porc_nulos_col); by = x -> x[2], rev = true)
    println(rpad(col, 30), ": ", round(pct, digits = 2), "%")
end

println("\nPorcentaje total de valores nulos en el dataset: ",
        round(porc_total_missing, digits = 2), "%")
println("===============================================")

=== Porcentaje de valores nulos por columna ===
tBodyGyroMag_mad_             : 10.03%
tBodyGyroMag_iqr_             : 10.03%
fBodyAcc_mad_Y                : 10.02%
fBodyAccJerk_mean_X           : 10.02%
fBodyBodyGyroMag_iqr_         : 10.0%
fBodyAcc_std_X                : 10.0%
tBodyAccMag_max_              : 10.0%
tGravityAccMag_std_           : 10.0%
tGravityAccMag_entropy_       : 10.0%
tBodyAccJerk_entropy_Y        : 10.0%
tBodyAccJerk_energy_X         : 10.0%
tBodyGyro_arCoeff_Y_2         : 9.99%
tBodyGyroJerk_arCoeff_Z_2     : 9.99%
fBodyAcc_maxInds_Y            : 9.99%
fBodyBodyGyroJerkMag_energy_  : 9.99%
fBodyGyro_mean_X              : 9.99%
fBodyGyro_maxInds_Z           : 9.99%
fBodyAccJerk_bandsEnergy_49_56_2: 9.99%
tBodyAcc_entropy_Z            : 9.99%
fBodyGyro_energy_Y            : 9.99%
tGravityAcc_std_Y             : 9.99%
fBodyAcc_kurtosis_Y           : 9.99%
tBodyAccJerkMag_arCoeff_2     : 9.99%
fBodyGyro_bandsEnergy_49_56_1 : 9.99%
tGravityAcc_max_X             : 9.

## 4. Imputación de valores ausentes (subject-wise)

En esta sección imputamos los valores faltantes de las variables numéricas siguiendo
un criterio **por sujeto**:

- Para cada sujeto (`subject`) se toman únicamente sus propias observaciones.
- Para cada columna numérica con valores ausentes:
  - Se comprueba si hay outliers mediante el rango intercuartílico (IQR).
  - Si **hay outliers**, se imputa con la **mediana**.
  - Si **no hay outliers**, se imputa con la **media**.
- No se modifican las columnas `subject` ni `Activity`.

El resultado es un nuevo dataset imputado que conserva la estructura original pero sin
valores faltantes en las variables numéricas.


In [7]:
# Cargar dataset consolidado desde data_processed
df = CSV.read("data_processed/dataset_consolidado.csv", DataFrame)


# -----------------------------------------------------------
# Función auxiliar para detectar columnas numéricas (permitiendo Missing)
# -----------------------------------------------------------
function col_contains_numeric(eltyp)
    if eltyp <: Real
        return true
    end
    try
        for t in Base.uniontypes(eltyp)
            if t <: Real
                return true
            end
        end
    catch
    end
    return false
end

# -----------------------------------------------------------
# Detección de outliers con IQR
# -----------------------------------------------------------
function tiene_outliers(vals)
    q1 = quantile(vals, 0.25)
    q3 = quantile(vals, 0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    any(x -> x < lower || x > upper, vals)
end

# -----------------------------------------------------------
# Imputación subject-wise
# -----------------------------------------------------------
function impute_subjectwise(df::DataFrame)
    df_imp = deepcopy(df)

    excluded = Set(["subject", "Activity"])   # no se imputan estas columnas

    # Detectar columnas numéricas a imputar
    numeric_cols = String[]
    for c in names(df_imp)
        if c ∉ excluded && col_contains_numeric(eltype(df_imp[!, c]))
            push!(numeric_cols, c)
        end
    end

    subjects = unique(df_imp.subject)

    println("Columnas numéricas a imputar: ", length(numeric_cols))
    println("Sujetos encontrados: ", length(subjects))

    for s in subjects
        rows_subject = df_imp.subject .== s

        for col in numeric_cols
            colvec = df_imp[rows_subject, col]
            nmiss = count(ismissing, colvec)
            if nmiss == 0
                continue
            end

            nonmiss = collect(skipmissing(colvec))
            if isempty(nonmiss)
                continue
            end

            method = tiene_outliers(nonmiss) ? "median" : "mean"
            value  = method == "median" ? median(nonmiss) : mean(nonmiss)

            mask = rows_subject .& ismissing.(df_imp[!, col])
            df_imp[mask, col] .= value
        end
    end

    return df_imp
end


# -----------------------------------------------------------
# Aplicar imputación y guardar resultado
# -----------------------------------------------------------
df_imputed = impute_subjectwise(df)

println("\nDataset imputado: ", nrow(df_imputed), " filas, ", ncol(df_imputed), " columnas.")

CSV.write("data_processed/dataset_consolidado_imputed.csv", df_imputed)

println("Guardado en: data_processed/dataset_consolidado_imputed.csv")


Columnas numéricas a imputar: 561
Sujetos encontrados: 30

Dataset imputado: 10299 filas, 563 columnas.
Guardado en: data_processed/dataset_consolidado_imputed.csv


## 5. Partición holdout (10 % de sujetos)

En este bloque se reserva un **10 % de los sujetos completos** como conjunto de
**test final (holdout)**, siguiendo las indicaciones del enunciado:

- Se parte del dataset ya imputado (`df_imputed`).
- Se obtienen todos los identificadores de sujetos (`subject`).
- Con la semilla `104` se selecciona aleatoriamente el 10 % de los sujetos.
- Todas las filas de esos sujetos pasan a formar el conjunto **test**.
- El resto de sujetos componen el conjunto **train**.

Este conjunto de test **no se utiliza en la validación cruzada** y se reserva
exclusivamente para la evaluación final de los modelos seleccionados.

In [8]:
# Cargar dataset imputado desde data_processed
df_imputed = CSV.read("data_processed/dataset_consolidado_imputed.csv", DataFrame)

println("Sujetos detectados en el dataset imputado:")
subjects = unique(df_imputed.subject)
println(subjects)

# Semilla pedida en el enunciado
Random.seed!(104)

# 10% de sujetos → al menos 1
n_test = max(1, round(Int, length(subjects) * 0.10))

println("Número total de sujetos: ", length(subjects))
println("Número de sujetos para TEST (10%): ", n_test)

# Selección aleatoria reproducible
test_subjects = Random.shuffle(subjects)[1:n_test]

println("\n=== Sujetos seleccionados para TEST (holdout) ===")
println(test_subjects)

# Máscaras de pertenencia
test_mask  = in.(df_imputed.subject, Ref(test_subjects))
train_mask = .!test_mask

# Particionar
df_test  = df_imputed[test_mask, :]
df_train = df_imputed[train_mask, :]

println("\nFilas train: ", nrow(df_train))
println("Filas test : ", nrow(df_test))

# ------------------ Guardado en data_processed ------------------

CSV.write("data_processed/dataset_train.csv", df_train)
CSV.write("data_processed/dataset_test.csv", df_test)

# Lista de sujetos del test
ts_df = DataFrame(subject = test_subjects)
CSV.write("data_processed/test_subjects.csv", ts_df)

# Informe del número de muestras por sujeto en el dataset completo
counts = combine(groupby(df_imputed, :subject), nrow => :n_rows)
CSV.write("data_processed/subject_counts.csv", counts)

println("\nArchivos guardados en data_processed/:")
println(" - Train dataset:        dataset_train.csv")
println(" - Test dataset:         dataset_test.csv")
println(" - Test subjects list:   test_subjects.csv")
println(" - Subject counts:       subject_counts.csv")


Sujetos detectados en el dataset imputado:
[1, 5, 7, 11, 3, 9, 23, 25, 15, 17, 21, 13, 19, 27, 29, 2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 26, 28, 30]
Número total de sujetos: 30
Número de sujetos para TEST (10%): 3

=== Sujetos seleccionados para TEST (holdout) ===
[25, 18, 22]

Filas train: 9205
Filas test : 1094

Archivos guardados en data_processed/:
 - Train dataset:        dataset_train.csv
 - Test dataset:         dataset_test.csv
 - Test subjects list:   test_subjects.csv
 - Subject counts:       subject_counts.csv


## 6. Validación cruzada individual-wise (5-Fold)

Tras aplicar la partición *holdout*, usamos únicamente el conjunto de entrenamiento
(`df_train`) para construir una validación cruzada 5-fold basada en **sujetos**:

- Cada fold contiene un subconjunto de sujetos completos.
- En cada fold, uno (o varios) sujetos se usan como **test interno**.
- El resto se usan como **train**.
- Nunca se mezclan instancias de un mismo sujeto entre train y test.
- Se fija la semilla `104` para reproducibilidad.

Este esquema es imprescindible porque los datos están fuertemente correlacionados por
sujeto; por tanto, una validación aleatoria estándar produciría *data leakage*.

In [9]:
# Cargar el conjunto de entrenamiento generado en el holdout
df_train = CSV.read("data_processed/dataset_train.csv", DataFrame)

# Sujetos disponibles en TRAIN
subjects_train = unique(df_train.subject)
n_subjects = length(subjects_train)
n_folds = 5

println("Sujetos disponibles en TRAIN: ", n_subjects)

# -----------------------------------------------------------
# Función generadora de folds balanceados
# -----------------------------------------------------------
function generate_subjectwise_folds(subjects::Vector, k::Int=5; seed=104)
    Random.seed!(seed)
    shuffled = Random.shuffle(subjects)

    base_size = div(length(shuffled), k)
    extra = mod(length(shuffled), k)

    folds = Vector{Vector{eltype(subjects)}}()
    start_idx = 1

    for i in 1:k
        fold_size = base_size + (i <= extra ? 1 : 0)
        push!(folds, shuffled[start_idx:start_idx+fold_size-1])
        start_idx += fold_size
    end

    return folds
end

folds = generate_subjectwise_folds(subjects_train, n_folds)

println("\nSujetos por fold:")
for i in 1:length(folds)
    println("Fold $i: ", folds[i])
end

# -----------------------------------------------------------
# Crear los CSV de train/test por fold
# -----------------------------------------------------------

# Crear la carpeta de salida si no existe
mkpath("data_processed/folds")

for i in 1:n_folds
    fold_subjects = folds[i]

    # Elegimos 1 sujeto como test interno (igual que tu script original)
    fold_subjects_shuffled = Random.shuffle(copy(fold_subjects))
    test_subject = fold_subjects_shuffled[1]        # sujeto de test interno
    train_subjects = fold_subjects_shuffled[2:end]  # resto son train

    println("\nFold $i")
    println("  Sujeto de test interno: ", test_subject)
    println("  Sujetos de train: ", train_subjects)

    test_mask  = in.(df_train.subject, Ref([test_subject]))
    train_mask = in.(df_train.subject, Ref(train_subjects))

    df_fold_train = df_train[train_mask, :]
    df_fold_test  = df_train[test_mask, :]

    # Guardar CSVs en data_processed/folds/
    CSV.write("data_processed/folds/fold$(i)_train.csv", df_fold_train)
    CSV.write("data_processed/folds/fold$(i)_test.csv",  df_fold_test)
end

println("\nValidación cruzada individual-wise 5-fold generada correctamente.")
println("Archivos guardados en data_processed/folds/")


Sujetos disponibles en TRAIN: 27

Sujetos por fold:
Fold 1: [15, 24, 28, 26, 29, 16]
Fold 2: [27, 2, 7, 9, 23, 21]
Fold 3: [10, 6, 30, 12, 4]
Fold 4: [3, 19, 17, 11, 1]
Fold 5: [8, 14, 5, 20, 13]

Fold 1
  Sujeto de test interno: 26
  Sujetos de train: [28, 24, 16, 15, 29]

Fold 2
  Sujeto de test interno: 2
  Sujetos de train: [9, 27, 23, 7, 21]

Fold 3
  Sujeto de test interno: 4
  Sujetos de train: [12, 30, 6, 10]

Fold 4
  Sujeto de test interno: 3
  Sujetos de train: [11, 19, 1, 17]

Fold 5
  Sujeto de test interno: 5
  Sujetos de train: [8, 14, 20, 13]

Validación cruzada individual-wise 5-fold generada correctamente.
Archivos guardados en data_processed/folds/


### 7. Normalización Min-Max con un nodo MLJ personalizado

La rúbrica de la práctica exige que la normalización se implemente como un **Nodo de MLJ**, y no como
una operación manual. El objetivo es garantizar que:

- El normalizador se ajusta **únicamente con el conjunto de entrenamiento**.
- El mismo transformador se aplica después sobre validaciones internas y sobre el conjunto de test.
- Se evita cualquier **fuga de información**.
- La normalización pueda integrarse dentro de un **pipeline de MLJ** o del proceso de validación cruzada.

En nuestro entorno concreto, los transformadores predefinidos de MLJ
(`Standardizer`, `FeatureRescaler`, `UnivariateStandardizer`, etc.) no estaban disponibles en la
versión de `MLJModels` instalada.  
Para seguir estrictamente la rúbrica, optamos por implementar un **nodo MLJ propio**, totalmente
compatible con la interfaz de MLJ.

Este nodo (`MyMinMaxScaler`) implementa:

- `fit(model, X)` → calcula los mínimos y máximos por columna únicamente a partir del conjunto *train*  
- `transform(model, X)` → aplica la fórmula del Min-Max scaling  
- Exclusión automática de columnas no numéricas  
- Ignora explícitamente las columnas `subject` y `Activity`, que no deben normalizarse  
- Es robusto ante valores `missing`  

Con este nodo se obtiene un funcionamiento equivalente al Min-Max tradicional,  
pero **respetando la filosofía y requisitos formales de MLJ**.

In [10]:
#-----------------------------------------------------------
# Definición del escalador Min-Max personalizado
#-----------------------------------------------------------
const MMI = MLJModelInterface

struct MyMinMaxScaler <: MMI.Unsupervised
    ignore::Vector{Symbol}
end

MyMinMaxScaler(; ignore = [:subject, :Activity]) = MyMinMaxScaler(ignore)

#-----------------------------------------------------------
# Implementación de fit y transform
#-----------------------------------------------------------
function MMI.fit(model::MyMinMaxScaler, verbosity::Int, X)

    # 1. columnas realmente numéricas
    numeric_cols = [
        c for c in names(X)
        if !(c in model.ignore) &&
           all(x -> x === missing || x isa Real, X[!, c])
    ]

    mins = Dict{String, Float64}()
    maxs = Dict{String, Float64}()

    for col in numeric_cols
        col_data = collect(skipmissing(X[!, col]))

        if isempty(col_data)
            mins[col] = 0.0
            maxs[col] = 0.0
        else
            mins[col] = minimum(col_data)
            maxs[col] = maximum(col_data)
        end
    end

    fitresult = (
        mins = mins,
        maxs = maxs,
        numeric_cols = numeric_cols
    )

    return fitresult, nothing, nothing
end


function MMI.transform(model::MyMinMaxScaler, fitresult, X)
    X_new = deepcopy(X)

    for col in fitresult.numeric_cols
        minv = fitresult.mins[col]
        maxv = fitresult.maxs[col]

        if maxv != minv
            X_new[!, col] = (X_new[!, col] .- minv) ./ (maxv - minv)
        else
            X_new[!, col] .= 0.0
        end
    end

    return X_new
end

#-----------------------------------------------------------
# Aplicar el escalado Min-Max y guardar los datasets escalados
#-----------------------------------------------------------

# Cargar los datasets de train y test desde data_processed
df_train = CSV.read("data_processed/dataset_train.csv", DataFrame)
df_test  = CSV.read("data_processed/dataset_test.csv", DataFrame)

scaler = MyMinMaxScaler(ignore = [:subject, :Activity])

mach = machine(scaler, df_train)
fit!(mach)

df_train_scaled = transform(mach, df_train)
df_test_scaled  = transform(mach, df_test)

CSV.write("data_processed/dataset_train_scaled.csv", df_train_scaled)
CSV.write("data_processed/dataset_test_scaled.csv", df_test_scaled)

[ Info: Training machine(MyMinMaxScaler(ignore = [:subject, :Activity]), …).


"data_processed/dataset_test_scaled.csv"

# MODELOS BÁSICOS Y SELECCIÓN DE ATRIBUTOS

### Selector sin reducción (baseline)

Este selector se usa como referencia: no elimina ninguna característica,
pero permite integrarlo como nodo MLJ en pipelines y compararlo con las demás técnicas
de selección de características.

In [4]:
using MLJModelInterface
const MMI = MLJModelInterface

# =====================================
# Selector de características: Sin reducción
# =====================================

"""
    SelectorNone()

Selector que no elimina ninguna característica.
Devuelve todas las columnas excepto :subject y :Activity.
"""
struct SelectorNone <: MMI.Unsupervised end

# Fit
function MMI.fit(model::SelectorNone, verbosity::Int, X)
    selected = setdiff(names(X), [:subject, :Activity])
    fitresult = (selected = selected,)
    return fitresult, nothing, (selected = selected,)
end

# Transform
function MMI.transform(model::SelectorNone, fitresult, X)
    return X[:, fitresult.selected]
end

### Selección de características ANOVA (F-test)

Para cada característica numérica se calcula un test ANOVA univariante tomando como
variable dependiente la clase `Activity`. Se seleccionan las 50 características con
mayor estadístico F. Este filtro se ajusta únicamente sobre los datos de entrenamiento
para evitar fuga de información.


In [5]:
using MLJModelInterface
const MMI = MLJModelInterface

using HypothesisTests
using CategoricalArrays
using Statistics

# ===============================
# Definición del modelo (Notebook)
# ===============================

"""
    SelectorANOVA(k = 50)

Selector de características basado en One-Way ANOVA.
Selecciona las k columnas con mayor estadístico F,
comparando cada feature entre niveles de Activity.
"""
struct SelectorANOVA <: MMI.Unsupervised
    k::Int
end

SelectorANOVA(; k = 50) = SelectorANOVA(k)


# ===============================
# Fit
# ===============================

function MMI.fit(model::SelectorANOVA, verbosity::Int, X)

    # 1) Asegurar target correcto
    @assert :Activity ∈ names(X) "El dataset debe contener la columna :Activity como target."

    y = X.Activity

    # Convertir Activity a categórica si es necesario
    if !(y isa CategoricalVector)
        y = categorical(y)
    end

    classes = levels(y)

    # 2) Seleccionar columnas de features
    feature_cols = Symbol[]
    for col in names(X)
        if col != :subject && col != :Activity
            push!(feature_cols, col)
        end
    end

    scores = Float64[]
    cols = Symbol[]

    # 3) Evaluar ANOVA por feature
    for col in feature_cols
        coldata = X[!, col]

        # Imputación de missing: media
        if any(ismissing, coldata)
            μ = mean(skipmissing(coldata))
            coldata = coalesce.(coldata, μ)
        end

        # Feature constante → no aporta
        if length(unique(coldata)) < 2
            push!(scores, -Inf)
            push!(cols, col)
            continue
        end

        # Construcción de grupos por clase
        groups = Vector{Vector{Float64}}()
        valid_feature = true

        for cls in classes
            vals = coldata[y .== cls]

            # ANOVA necesita al menos 2 observaciones por grupo
            if length(vals) <= 1
                valid_feature = false
                break
            end

            push!(groups, collect(vals))
        end

        if !valid_feature
            push!(scores, -Inf)
            push!(cols, col)
            continue
        end

        # ANOVA
        try
            test = OneWayANOVA(groups...)
            push!(scores, test.F)
        catch
            push!(scores, -Inf)
        end

        push!(cols, col)
    end

    # 4) Ranking y selección final
    order = sortperm(scores, rev = true)
    k = min(model.k, length(cols))
    selected = cols[order][1:k]

    fitresult = (selected = selected,)
    report = (scores = scores, selected = selected)

    return fitresult, nothing, report
end


# ===============================
# Transform
# ===============================

function MMI.transform(model::SelectorANOVA, fitresult, X)
    return X[:, fitresult.selected]
end


### Filtrado de características mediante correlación de Pearson

PROBLEMA: Pearson necesita que todo sea numérico, pero nuestra target es categórica (se prueba igual porque el señor lo pide)

In [6]:
using MLJModelInterface
const MMI = MLJModelInterface

using Statistics
using CategoricalArrays

# ==============================
# Definición del modelo (Notebook)
# ==============================

"""
    SelectorPearson(k = 50)

Selector de características basado en correlación de Pearson.
Convierte Activity a índices numéricos y calcula |r|
entre cada feature y la variable objetivo.
"""
struct SelectorPearson <: MMI.Unsupervised
    k::Int
end

SelectorPearson(; k = 50) = SelectorPearson(k)


# ==============================
# Fit
# ==============================

function MMI.fit(model::SelectorPearson, verbosity::Int, X)

    # 1) Preparar Activity
    @assert :Activity ∈ names(X)

    ycat = X.Activity
    if !(ycat isa CategoricalVector)
        ycat = categorical(ycat)
    end

    # Convertir actividades a índices numéricos (1..C)
    y = levelcode.(ycat)

    # 2) Seleccionar columnas de features
    feature_cols = Symbol[]
    for col in names(X)
        if col != :subject && col != :Activity
            push!(feature_cols, col)
        end
    end

    scores = Float64[]
    cols = Symbol[]

    # 3) Pearson por feature
    for col in feature_cols
        x = X[!, col]

        # Si es constante → no aporta
        if length(unique(skipmissing(x))) < 2
            push!(scores, 0.0)
            push!(cols, col)
            continue
        end

        # Alinear missing
        mask = .!ismissing.(x)
        x_clean = Float64.(skipmissing(x))
        y_clean = y[mask]

        # Pocos datos → score 0
        if length(x_clean) < 2
            push!(scores, 0.0)
            push!(cols, col)
            continue
        end

        # Calcular correlación
        try
            r = cor(x_clean, y_clean)
            push!(scores, abs(r))
        catch
            push!(scores, 0.0)
        end

        push!(cols, col)
    end

    # 4) Ordenar y seleccionar
    order = sortperm(scores, rev=true)
    k = min(model.k, length(cols))
    selected = cols[order][1:k]

    fitresult = (selected = selected,)
    report = (scores = scores, selected = selected)

    return fitresult, nothing, report
end

# ==============================
# Transform
# ==============================

function MMI.transform(model::SelectorPearson, fitresult, X)
    return X[:, fitresult.selected]
end


### Filtrado Spearman

Spearman mide la correlación por rangos. Es más robusto ante relaciones no lineales.
Se seleccionan las 50 características con mayor |ρ|.


In [7]:
using MLJModelInterface
const MMI = MLJModelInterface

using Statistics
using StatsBase
using CategoricalArrays

# ==============================
# Definición del modelo (Notebook)
# ==============================

"""
    SelectorSpearman(k = 50)

Selector de características basado en correlación de Spearman.
Convierte Activity en códigos numéricos y calcula |ρ|
entre cada feature y la variable objetivo.
"""
struct SelectorSpearman <: MMI.Unsupervised
    k::Int
end

SelectorSpearman(; k=50) = SelectorSpearman(k)


# ==============================
# Fit
# ==============================

function MMI.fit(model::SelectorSpearman, verbosity::Int, X)

    @assert :Activity ∈ names(X)

    # Convertir Activity a categórica si hace falta
    ycat = X.Activity
    if !(ycat isa CategoricalVector)
        ycat = categorical(ycat)
    end

    # Convertir categorías a códigos numéricos 1..C
    y = levelcode.(ycat)

    # Features válidos
    feature_cols = Symbol[]
    for col in names(X)
        if col != :subject && col != :Activity
            push!(feature_cols, col)
        end
    end

    scores = Float64[]
    cols = Symbol[]

    # Calcular Spearman por feature
    for col in feature_cols
        x = X[!, col]

        # Quitar missing alineados
        mask = .!ismissing.(x)
        x_clean = Float64.(skipmissing(x))
        y_clean = y[mask]

        # Si el feature es constante o insuficiente → score 0
        if length(unique(x_clean)) < 2 || length(x_clean) < 2
            push!(scores, 0.0)
            push!(cols, col)
            continue
        end

        # Intentar correlación
        try
            r = corspearman(x_clean, y_clean)
            push!(scores, abs(r))
        catch
            push!(scores, 0.0)
        end

        push!(cols, col)
    end

    # Ordenar y seleccionar top-k
    order = sortperm(scores, rev=true)
    k = min(model.k, length(cols))
    selected = cols[order][1:k]

    fitresult = (selected = selected,)
    report = (scores = scores, selected = selected)

    return fitresult, nothing, report
end


# ==============================
# Transform
# ==============================

function MMI.transform(model::SelectorSpearman, fitresult, X)
    return X[:, fitresult.selected]
end


### Filtrado Kendall Tau

El coeficiente Tau de Kendall es un estimador no paramétrico basado en concordancias
y discordancias entre pares. Resulta más estable con datos con ruido.

Se seleccionan las 50 mejores características por |τ|.


In [8]:
using MLJModelInterface
const MMI = MLJModelInterface

using Statistics
using StatsBase
using CategoricalArrays

# ==============================
# Definición del modelo (Notebook)
# ==============================

"""
    SelectorKendall(k = 50)

Selector de características basado en correlación de Kendall Tau.
Convierte Activity a códigos numéricos y calcula |τ|
entre cada feature y la variable objetivo.
"""
struct SelectorKendall <: MMI.Unsupervised
    k::Int
end

SelectorKendall(; k = 50) = SelectorKendall(k)


# ==============================
# Fit
# ==============================

function MMI.fit(model::SelectorKendall, verbosity::Int, X)

    @assert :Activity ∈ names(X)

    # Convertir Activity a categórica si fuese necesario
    ycat = X.Activity
    if !(ycat isa CategoricalVector)
        ycat = categorical(ycat)
    end

    # Convertir categorías a índices numéricos 1..C
    y = levelcode.(ycat)

    # Seleccionar columnas de features válidas
    feature_cols = Symbol[]
    for col in names(X)
        if col != :subject && col != :Activity
            push!(feature_cols, col)
        end
    end

    scores = Float64[]
    cols = Symbol[]

    # Calcular Kendall Tau para cada feature
    for col in feature_cols
        x = X[!, col]

        # Alinear missing
        mask = .!ismissing.(x)
        x_clean = Float64.(skipmissing(x))
        y_clean = y[mask]

        # Feature constante o con muy pocos datos → score 0
        if length(unique(x_clean)) < 2 || length(x_clean) < 2
            push!(scores, 0.0)
            push!(cols, col)
            continue
        end

        # Kendall Tau
        try
            τ = corkendall(x_clean, y_clean)
            push!(scores, abs(τ))
        catch
            push!(scores, 0.0)
        end

        push!(cols, col)
    end

    # Ordenar y seleccionar las k mejores
    order = sortperm(scores, rev=true)
    k = min(model.k, length(cols))
    selected = cols[order][1:k]

    fitresult = (selected=selected,)
    report = (scores=scores, selected=selected)

    return fitresult, nothing, report
end


# ==============================
# Transform
# ==============================

function MMI.transform(model::SelectorKendall, fitresult, X)
    return X[:, fitresult.selected]
end


### Filtrado por Información Mutua (MI)

La información mutua permite capturar dependencias no lineales entre cada característica
y la variable objetivo. Dado que la versión de MLJBase instalada no incluye la función
`mutualinfo`, se ha implementado una versión personalizada basada en histogramas, que 
calcula MI de forma robusta sin dependencias externas.

De cada característica se obtiene su MI con la clase, y se seleccionan las 50 con mayor valor.

In [9]:
"""
    mutual_information(x, y; bins=10)

Calcula la información mutua entre dos vectores x e y.
Si x o y son continuos, se discretizan automáticamente en `bins` estratos.
Se ignoran valores missing de manera alineada.
"""
function mutual_information(x, y; bins=10)

    # 1. Eliminar missing alineados
    mask = .!(ismissing.(x) .| ismissing.(y))
    x = x[mask]
    y = y[mask]

    # 2. Discretizar si son continuos
    if eltype(x) <: Real
        hx = fit(Histogram, x, bins)
        x = StatsBase.binindex.(Ref(hx), x)
    end

    if eltype(y) <: Real
        hy = fit(Histogram, y, bins)
        y = StatsBase.binindex.(Ref(hy), y)
    end

    # 3. Contar probabilidades
    n = length(x)
    px  = countmap(x)
    py  = countmap(y)
    pxy = countmap(zip(x, y))

    # 4. Calcular MI correctamente
    mi = 0.0
    for ((xi, yi), nxy) in pxy
        px_i = px[xi]
        py_i = py[yi]

        pxy_p = nxy / n
        px_p  = px_i / n
        py_p  = py_i / n

        mi += pxy_p * log(pxy_p / (px_p * py_p + eps()))
    end

    return mi
end

mutual_information

In [10]:
using MLJModelInterface
const MMI = MLJModelInterface

using Statistics
using CategoricalArrays

# ==============================
# Definición del modelo
# ==============================

"""
SelectorMI(k=50)

Selector de características basado en Información Mutua.
Selecciona los k features con mayor MI respecto a Activity.
"""
struct SelectorMI <: MMI.Unsupervised
    k::Int
end

SelectorMI(; k=50) = SelectorMI(k)

# ==============================
# Fit
# ==============================

function MMI.fit(model::SelectorMI, verbosity::Int, X)

    @assert :Activity ∈ names(X)

    # Convertir Activity a vector numérico
    ycat = X.Activity
    if !(ycat isa CategoricalVector)
        ycat = categorical(ycat)
    end
    y = levelcode.(ycat)

    # Identificar columnas de features
    feature_cols = Symbol[]
    for col in names(X)
        if col != :subject && col != :Activity
            push!(feature_cols, col)
        end
    end

    scores = Float64[]
    cols = Symbol[]

    # Calcular MI para cada feature
    for col in feature_cols
        x = X[!, col]

        try
            mi = mutual_information(x, y)
            push!(scores, mi)
        catch
            push!(scores, 0.0)
        end

        push!(cols, col)
    end

    # Selección top-k
    order = sortperm(scores, rev=true)
    k = min(model.k, length(cols))
    selected = cols[order][1:k]

    fitresult = (selected = selected,)
    report = (scores = scores, selected = selected)

    return fitresult, nothing, report
end

# ==============================
# Transform
# ==============================

function MMI.transform(model::SelectorMI, fitresult, X)
    return X[:, fitresult.selected]
end

### Filtrado RFE (Recursive Feature Elimination) con Regresión Logística

El enunciado especifica que el método RFE debe utilizar una regresión logística,
eliminando el 50 % de las características en cada iteración. 

Dado que la versión de MLJModels disponible en este entorno no incluye un modelo
de regresión logística, se ha implementado el RFE mediante `GLM.jl`, el paquete
estándar de Julia para modelos lineales generalizados.

El procedimiento es el siguiente:

1. Se toma el conjunto completo de características numéricas.
2. Se ajusta una regresión logística (`glm`) con todas ellas.
3. Se ordenan las características según la magnitud absoluta de sus coeficientes.
4. Se elimina el 50 % menos relevante.
5. El proceso se repite hasta conservar exactamente **50 características**.

Este nodo sigue estrictamente la rúbrica y el enunciado, 
y se integra en MLJ mediante la interfaz `fit` → `transform`.


In [45]:
using FeatureSelection
using MLJLinearModels  # para logistic regression
RFE = @load RecursiveFeatureElimination pkg=FeatureSelection verbosity=0


const MMI = MLJModelInterface

# ==============================
# Definición del selector RFE
# ==============================

"""
    SelectorRFE(k=50)

Selector de características basado en RFE (eliminación recursiva de características),
usando LogisticClassifier como estimador base, eliminando el 50% de features en cada iteración,
hasta dejar k.
"""
struct SelectorRFE <: MMI.Unsupervised
    k::Int
end

SelectorRFE(; k=50) = SelectorRFE(k)

# ==============================
# Fit (entrenamiento del selector)
# ==============================

function MMI.fit(model::SelectorRFE, verbosity::Int, X)

    @assert :Activity ∈ names(X)

    # Target
    y = X.Activity

    # Features válidos
    feature_cols = filter(c -> c ∉ (:subject, :Activity), names(X))

    # Modelo base para RFE (logistic classifier compatible MLJ)
    base_model = LogisticClassifier(penalty=:l2, lambda=1.0)

    # Construcción del modelo RFE
    rfe_model = RecursiveFeatureElimination(
        model = base_model,
        features = feature_cols,
        k = model.k,            # número de features finales
        frac = 0.5              # elimina 50% por iteración
    )

    # Entrenar la selección
    mach = machine(rfe_model, X, y)
    fit!(mach, verbosity=verbosity)

    selected = report(mach).selected_features

    fitresult = (selected = selected,)
    report_out = (selected = selected,)

    return fitresult, nothing, report_out
end

# ==============================
# Transform
# ==============================

function MMI.transform(model::SelectorRFE, fitresult, X)
    return X[:, fitresult.selected]
end

## Proyecciones (Sin projección, PCA, LDA, ICA)

In [46]:
#IMPLEMENTACIÓN PROPIA MAYBE NO VA
using MLJModelInterface
const MMI = MLJModelInterface

struct NoProjection <: MMI.Unsupervised end

# Fit que no hace nada
function MMI.fit(model::NoProjection, verbosity::Int, X)
    return (nothing,), nothing, nothing

end

# Transform que devuelve X tal cual
function MMI.transform(model::NoProjection, fitresult, X)
    return X
end

In [14]:
PCA = @load PCA pkg=MultivariateStats verbosity=0

MLJMultivariateStatsInterface.PCA

In [15]:
LDA = @load LDA pkg=MultivariateStats verbosity=0

MLJMultivariateStatsInterface.LDA

In [16]:
ICA = @load ICA pkg=MultivariateStats verbosity=0

MLJMultivariateStatsInterface.ICA

## MODELOS DE CLASIFICACIÓN

In [17]:
NeuralNetworkClassifier = @load NeuralNetworkClassifier pkg=MLJFlux verbosity=0

MLJFlux.NeuralNetworkClassifier

In [18]:
KNN = @load KNNClassifier pkg=NearestNeighborModels verbosity=0

NearestNeighborModels.KNNClassifier

In [19]:
SVC =@load SVC pkg=LIBSVM verbosity=0

MLJLIBSVMInterface.SVC

## PIPELINES

| ID | Filtro de características       | Proyección                       | Clasificador | Hiperparámetros            |
| -- | ------------------------------- | -------------------------------- | ------------ | -------------------------- |
| 1  | **Sin filtrado** (SelectorNone) | **Sin reducción** (NoProjection) | MLP          | arquitectura **[50]**      |
| 2  | **ANOVA**                       | **PCA**                          | MLP          | arquitectura **[100]**     |
| 3  | **Mutual Information**          | **ICA**                          | MLP          | arquitectura **[100, 50]** |
| 4  | **Pearson**                     | **LDA**                          | KNN          | **k = 1**                  |
| 5  | **Spearman**                    | **Sin reducción** (NoProjection) | KNN          | **k = 10**                 |
| 6  | **Kendall Tau**                 | **PCA**                          | KNN          | **k = 20**                 |
| 7  | **RFE (Logistic Regression)**   | **ICA**                          | SVM          | **C = 0.1**                |
| 8  | **ANOVA**                       | **LDA**                          | SVM          | **C = 0.5**                |
| 9  | **Sin filtrado** (SelectorNone) | **PCA**                          | SVM          | **C = 1.0**                |


In [47]:
pipe1 = Pipeline(
    steps = [
        :filter => SelectorNone(),
        :proj   => NoProjection(),
        :clf    => NeuralNetworkClassifier(builder=MLJFlux.MLP(hidden=(50,)))
    ],
    name = :Pipe1
)

StaticPipeline(
  steps = Pair{Symbol, Model}[:filter => SelectorNone(), :proj => NoProjection(), :clf => NeuralNetworkClassifier(builder = MLP(hidden = (50,), …), …)], 
  name = :Pipe1, 
  cache = true)

In [26]:
pipe2 = Pipeline(
    steps = [
        :filter => SelectorANOVA(k=50),
        :proj   => PCA(maxoutdim=20),
        :clf    => NeuralNetworkClassifier(builder=MLJFlux.MLP(hidden=(100,)))
    ],
    name = :Pipe2
)

StaticPipeline(
  steps = Pair{Symbol, Model}[:filter => SelectorANOVA(k = 50), :proj => PCA(maxoutdim = 20, …), :clf => NeuralNetworkClassifier(builder = MLP(hidden = (100,), …), …)], 
  name = :Pipe2, 
  cache = true)

In [27]:
pipe3 = Pipeline(
    steps = [
        :filter => SelectorMI(k=50),
        :proj   => ICA(outdim=20),
        :clf    => NeuralNetworkClassifier(builder=MLJFlux.MLP(hidden=(100, 50)))
    ],
    name = :Pipe3
)

StaticPipeline(
  steps = Pair{Symbol, Model}[:filter => SelectorMI(k = 50), :proj => ICA(outdim = 20, …), :clf => NeuralNetworkClassifier(builder = MLP(hidden = (100, 50), …), …)], 
  name = :Pipe3, 
  cache = true)

In [28]:
pipe4 = Pipeline(
    steps = [
        :filter => SelectorPearson(k=50),
        :proj   => LDA(),
        :clf    => KNN(K=1)
    ],
    name = :Pipe4
)

StaticPipeline(
  steps = Pair{Symbol, Model}[:filter => SelectorPearson(k = 50), :proj => LDA(method = gevd, …), :clf => KNNClassifier(K = 1, …)], 
  name = :Pipe4, 
  cache = true)

In [48]:
pipe5 = Pipeline(
    steps = [
        :filter => SelectorSpearman(k=50),
        :proj   => NoProjection(),
        :clf    => KNN(K=10)
    ],
    name = :Pipe5
)

StaticPipeline(
  steps = Pair{Symbol, Model}[:filter => SelectorSpearman(k = 50), :proj => NoProjection(), :clf => KNNClassifier(K = 10, …)], 
  name = :Pipe5, 
  cache = true)

In [30]:
pipe6 = Pipeline(
    steps = [
        :filter => SelectorKendall(k=50),
        :proj   => PCA(maxoutdim=20),
        :clf    => KNN(K=20)
    ],
    name = :Pipe6
)

StaticPipeline(
  steps = Pair{Symbol, Model}[:filter => SelectorKendall(k = 50), :proj => PCA(maxoutdim = 20, …), :clf => KNNClassifier(K = 20, …)], 
  name = :Pipe6, 
  cache = true)

In [43]:
pipe7 = Pipeline(
    steps = [
        :filter => SelectorRFE(k=50),
        :proj   => ICA(outdim=20),
        :clf    => SVC(cost=0.1)
    ],
    name = :Pipe7
)

StaticPipeline(
  steps = Pair{Symbol, Model}[:filter => SelectorRFE(k = 50), :proj => ICA(outdim = 20, …), :clf => SVC(kernel = RadialBasis, …)], 
  name = :Pipe7, 
  cache = true)

In [34]:
pipe8 = Pipeline(
    steps = [
        :filter => SelectorANOVA(k=50),
        :proj   => LDA(),
        :clf    => SVC(cost=0.5)
    ],
    name = :Pipe8
)

StaticPipeline(
  steps = Pair{Symbol, Model}[:filter => SelectorANOVA(k = 50), :proj => LDA(method = gevd, …), :clf => SVC(kernel = RadialBasis, …)], 
  name = :Pipe8, 
  cache = true)

In [35]:
pipe9 = Pipeline(
    steps = [
        :filter => SelectorNone(),
        :proj   => PCA(maxoutdim=20),
        :clf    => SVC(cost=1.0)
    ],
    name = :Pipe9
)

StaticPipeline(
  steps = Pair{Symbol, Model}[:filter => SelectorNone(), :proj => PCA(maxoutdim = 20, …), :clf => SVC(kernel = RadialBasis, …)], 
  name = :Pipe9, 
  cache = true)